In [1]:
%load_ext autoreload
%autoreload 2

In [4]:
import sys
import os
sys.path.append(os.path.join(os.getcwd(), '../'))

In [9]:
import pandas as pd
import warnings

import json
from numpy import random

from model.optimization import load_model, eval_model
from model.dataset import get_dataframe

import lightgbm as lgb

DEFAULT_RANDOM_SEED = 774
random.mtrand._rand.seed(DEFAULT_RANDOM_SEED)
warnings.filterwarnings("ignore")

category="min_tpm_5"

In [8]:
combined_df = pd.read_csv("../../data/genes.csv").fillna(0)
extra_data_df = pd.read_csv("../../data/extra_data.tsv", delimiter="\t", keep_default_na=False)
final_df = pd.merge(combined_df, extra_data_df, on="sample_id", how="left")

In [11]:
importances = json.loads(open(f"../../preprocessed/{category}/important_genes_random_forest_f1.json").readline())
gene_count_by_subtype = json.loads(open(f"../../preprocessed/{category}/gene_count_by_subtype_random_forest_f1.json").readline())

In [31]:

for subtype in importances:
  clustered_subtype_df = final_df.copy()
  clustered_subtype_df["matching_subtype"] = clustered_subtype_df["subtype"] == subtype
  print(f"===== {subtype} =====")
  for sex in importances[subtype]:
    print(f"===== {sex} =====")
    filtered_sex_df = clustered_subtype_df[clustered_subtype_df["sex"] == sex].copy()
    for gene in importances[subtype][sex][:gene_count_by_subtype[subtype]]:
      average_matching = filtered_sex_df[filtered_sex_df["matching_subtype"]][gene["gene"]].agg(["mean", "std"]).to_dict()
      average_not_matching = filtered_sex_df[~filtered_sex_df["matching_subtype"]][gene["gene"]].agg(["mean", "std"]).to_dict()
      print(f"{gene['gene']} - Importance: {'%.04f' % gene['importance']}")
      print(f"Subtype expression: {'%.04f' % average_matching['mean']} ± {'%.04f' % average_matching['std']}")
      print(f"Not-subtype expression: {'%.04f' % average_not_matching['mean']} ± {'%.04f' % average_not_matching['std']}")
      print(filtered_sex_df[filtered_sex_df["matching_subtype"]][gene["gene"]].sort_values())
      print()

===== BCRABL1 =====
===== Male =====
WNT9A - Importance: 0.0116
Subtype expression: 24.4645 ± 25.6033
Not-subtype expression: 1.7176 ± 5.7786
1      0.81
14     0.84
33     1.26
12     1.45
31     1.64
13     2.52
3      3.78
11     8.89
37     9.89
36    10.83
7     19.13
10    19.79
4     20.28
35    27.99
0     28.06
34    30.36
15    33.74
21    34.95
8     48.02
25    73.42
17    79.32
20    81.25
Name: WNT9A, dtype: float64

PTPRO - Importance: 0.0078
Subtype expression: 6.7936 ± 6.4432
Not-subtype expression: 1.3264 ± 2.1999
13     0.83
12     1.32
36     2.04
0      2.21
37     2.30
3      2.50
31     2.97
14     3.59
10     3.60
1      4.24
11     4.99
33     5.47
15     7.01
35     7.05
25     7.14
4      7.16
34     7.61
20     8.08
7     10.62
21    11.88
17    17.06
8     29.79
Name: PTPRO, dtype: float64

TBXAS1 - Importance: 0.0068
Subtype expression: 23.1659 ± 16.2933
Not-subtype expression: 6.5017 ± 7.7282
14     1.00
12     3.32
11     5.13
33     8.78
20     9.28
37 